# Model Testing

Scripts for testing each component of the defense network and the overall network performance.

In [ ]:
# Library Imports.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import tensorflow as tf
# import os
import sys


# Class/Function Imports
from SVM import PerturbationDetectorSVM
current_dir = os.getcwd()
target_dir = os.path.join(current_dir, 'trapdoor', 'trapdoor')
trapdoor_path = os.path.abspath(target_dir)
if trapdoor_path not in sys.path:
    sys.path.append(trapdoor_path)

In [ ]:
# Import models
prn = tf.keras.models.load_model('models/trained_prn_res.h5', compile=False)
prn.is_trained = True
honeypot_model = tf.keras.models.load_model('models/cifar_model.h5', compile=False)
joint_model = tf.keras.models.load_model('models/trained_joint_model_res.h5', compile=False)
joint_model.is_trained = True

svm_detector = PerturbationDetectorSVM(prn, device=None, keep_coeffs=8) 
svm_detector.pipeline = joblib.load('models/svm_detector_res.joblib')
svm_detector.is_trained = True

In [ ]:
# Import data
mixed_test_X = np.load('./data/mixed_test_X.npy')
clean_test_X = np.load('./data/clean_test_X.npy')
clean_test_Y = np.load('./data/mixed_test_Y.npy')
test_attack_labels = np.load('./data/test_attack_labels.npy')
mixed_train_X = np.load('./data/mixed_train_X.npy')
train_attack_labels = np.load('./data/train_attack_labels.npy')

In [ ]:
def evaluate_baseline_honeypot(honeypot_model, clean_x, mixed_x, clean_y):
    """
    Evaluates how vulnerable the raw Honeypot model is to UAPs 
    WITHOUT the PRN defense layer.
    """
    print("\n" + "="*40)
    print(" BASELINE HONEYPOT EVALUATION")
    print("="*40)

    clean_preds = honeypot_model.predict(clean_x, batch_size=64)
    true_labels = np.argmax(clean_y, axis=1) if len(clean_y.shape) > 1 else clean_y
    pred_labels_clean = np.argmax(clean_preds, axis=1)
    
    clean_acc = accuracy_score(true_labels, pred_labels_clean)
    print(f"Honeypot Accuracy on CLEAN test data: {clean_acc * 100:.2f}%")

    mixed_preds = honeypot_model.predict(mixed_x, batch_size=64)
    pred_labels_mixed = np.argmax(mixed_preds, axis=1)
    
    mixed_acc = accuracy_score(true_labels, pred_labels_mixed)
    print(f"Honeypot Accuracy on MIXED (UAP) test data: {mixed_acc * 100:.2f}%")
    
    if clean_acc > 0:
        fooling_rate = 1.0 - (mixed_acc / clean_acc)
        print(f"Effective UAP Fooling Rate: {fooling_rate * 100:.2f}%")
    print("="*40 + "\n")

In [ ]:
tf.compat.v1.disable_eager_execution()
print("Evaluating Normal Honeypot Model")

evaluate_baseline_honeypot(honeypot_model, clean_test_X, mixed_test_X, clean_test_Y)

# Evaluates the model WITH the defense architecture.
print("Evaluating Defense Architecture")
svm_preds = svm_detector.predict(mixed_test_X) 

final_preds = []

for i in range(len(mixed_test_X)):
    img = mixed_test_X[i:i+1]
    
    if svm_preds[i] == 0:
        pred = honeypot_model.predict(img)
    else:
        pred = joint_model.predict(img)
    
        
    final_preds.append(np.argmax(pred))

final_preds = np.array(final_preds)
actual_labels = np.argmax(clean_test_Y, axis=1)

system_acc = np.mean(final_preds == actual_labels)
print(f"Full System Accuracy: {system_acc * 100:.2f}%")

print("\n--- Detection Performance (SVM) ---")
print(classification_report(test_attack_labels, svm_preds, target_names=['Clean', 'Attack']))

clean_idx = np.where(test_attack_labels == 0)[0]
uap_idx = np.where(test_attack_labels == 1)[0]

clean_system_acc = np.mean(final_preds[clean_idx] == actual_labels[clean_idx])
uap_system_acc = np.mean(final_preds[uap_idx] == actual_labels[uap_idx])
if clean_system_acc > 0:
    pipeline_fooling_rate = (clean_system_acc - uap_system_acc) / clean_system_acc
else:
    pipeline_fooling_rate = 0.0


print("\n--- Defended Pipeline (PRN+SVM) vs UAP ---")
print(f"Pipeline Accuracy on CLEAN test data: {clean_system_acc * 100:.2f}%")
print(f"Pipeline Accuracy on UAP test data:   {uap_system_acc * 100:.2f}%")
print(f"Effective UAP Fooling Rate:           {pipeline_fooling_rate * 100:.2f}%")